# Ventricular tachyarrhythmias: detection pipeline

Working notebook for the paper's experiments, modeled on `vfpred/codes/VFPred.ipynb`.
It runs our steps end to end and holds results as each phase lands. See `paper/PLAN.md`
for the full plan and `paper/PAPER.md` for the manuscript.

Unlike VFPred, this pipeline is classical and deterministic: no SVM, no CNN. The candidate
detectors are TCSC, VFLEAK, SPEC, HILB, and MEA, compared on shockable vs non-shockable and
judged on discrimination and compute cost.

Dataset decisions in force (PLAN.md, Phase 2): read through `pxg.cbor`, filter once per
record, slide overlapping 1 s-step windows at 8 s and 4 s, write per-record TSVs under
`data/s<sec>/`. Feature extraction feeds vftx a millivolt signal (`int / 200`) with vftx's
own filtering off (`apply_filters=False`); QRS features are skipped until the exg-core beat
annotations are wired in.

## Setup

In [ ]:
import numpy as np
import pandas as pd

from vfta import build, labels

# pxg is installed in the conda env. Run this notebook from the repo root so
# `from vfta import ...` and the default work/ (cbor) and data/ paths resolve.

DBS = ['mitdb', 'cudb', 'vfdb', 'ahadb']
WINDOWS = [8.0, 4.0]

## Phase 2: build the dataset (per-record TSV)

`cbor record -> record-level filtering -> sliding window -> data/s<sec>/<db>/<rid>.tsv`,
parallel by record. The build is heavy, so run it once per database and window; the cells
below are left commented to avoid rerunning by accident.

In [ ]:
# for win in WINDOWS:
#     for db in DBS:
#         res = build.build_database(db, window_sec=win, step_sec=1.0, jobs=-1)
#         print(f'{db} s{int(win)}: {len(res)} records, {sum(n for _, n in res)} windows')
#     pass #for
# pass #for

## Phase 2: label windows (shockable / VT / VFL / VF)

Each window gets a `Rhythm` from its dominant clean episode (>= 90% of the window) and a
binary `Shock` label. Windows below the purity threshold are `MIX` (evaluation only).

In [ ]:
df = labels.load_dataset(['vfdb', 'cudb'], window_sec=8.0, purity=0.9)
print(df['Rhythm'].value_counts())
print(df['Shock'].value_counts())

## Phase 2: feature extraction (vftx)

`build_record` / `build_database` now compute the vftx feature columns per window. Two
families are excluded by design (see `paper/PLAN.md`):

- QRS features (RR, beat ratios): a QRS detector must blank during VF/VFL, so a signal-only
  VF/VFL detector cannot depend on it without circularity.
- EMD IMF-LZ: about 2 s/window, too slow for the real-time target.

So the bulk build writes 16 cheap features (~9 ms/window). Sample entropy (~55 ms/window)
is opt-in via `--spen` / `spen=True`. The signal is converted to millivolts by `int / 200`
and vftx's own frequency filtering is off (the record is already filtered).

In [ ]:
from vfta import features

print('feature columns:', features.feature_names())
print('with sample entropy:', features.feature_names(spen=True)[-1])

# build with features (already wired into build_record/build_database):
# build.build_database('mitdb', window_sec=8.0, jobs=-1)            # 16 cheap features
# build.build_database('mitdb', window_sec=8.0, jobs=-1, spen=True) # + sample entropy

## Phase 3: dataset composition

Window counts per database, class, and window length (Results 4.1).

In [ ]:
# TODO: composition table across all databases and both windows

## Phase 3: feature screen

Score each of the 27 features against the shockable label: point-biserial correlation,
mutual information, single-feature AUC. Flag redundancy with a feature-feature correlation
heatmap. This confirms the two added candidates (HILB, MEA).

In [ ]:
# TODO: per-feature correlation / mutual information / AUC ranking

## Phase 3: candidate shootout

Five detectors (TCSC, VFLEAK, SPEC, HILB, MEA): discrimination plus a rough compute cost per
window, at 8 s, leaders rerun at 4 s. The winner is chosen by discrimination weighed against
cost, not the top score alone.

In [ ]:
# TODO: candidate detectors (threshold + decision) and the discrimination-vs-cost table

## Phase 4: winner tuning

Sweep the winning feature's threshold, build the ROC curve, pick operating points; report
F1, Se, Sp, PPV, Acc, G-Mean under the three VFL configurations and both windows.

In [ ]:
# TODO: ROC sweep and operating-point table for the winner (hypothesis: TCSC)

## Phase 4: flutter vs fibrillation (winner only)

Whether the winning feature separates VFL from VF; if not, fall back to the IMF-LZ features
(LZ on EMD modes), which Hong added for the VF-vs-VT distinction.

In [ ]:
# TODO: VFL-vs-VF separation for the winning feature